In [ ]:
from transformers import (
    AutoTokenizer)

In [ ]:
MODEL_ID = "TurkuNLP/web-register-classification-multilingual-bge"


In [ ]:
from pathlib import Path
OUTPUT_DIR = Path(
    r"/scratch/project_462001491/nima/Hybrid_SP_ID_effect/Dataset_Hugging_face/without_NA/_MultiCore_BGE_m3_finetuned_evaluation/Combined_hybrid_no_NA"
)

In [ ]:
import json
from pathlib import Path

model_dir = Path(
    OUTPUT_DIR / "final_model"
)

with open(
    model_dir / "config.json",
    "r",
    encoding="utf-8",
) as f:
    config = json.load(f)

print(json.dumps(config, indent=2))


In [ ]:
AutoTokenizer.from_pretrained(model_dir)


In [ ]:
from transformers import AutoModelForSequenceClassification

saved_model = AutoModelForSequenceClassification.from_pretrained(
    OUTPUT_DIR / "final_model"
)

print(saved_model)
print(saved_model.config)
print(saved_model.base_model_prefix)


In [ ]:
print(best_model)
print(best_model.config)

In [ ]:
print(best_model.base_model_prefix)


# Extracting fine-tuned embed

In [ ]:
import numpy as np
import torch

In [ ]:
output_hidden_states=True


In [ ]:
def extract_embeddings(model, dataset, batch_size=16):

    model.eval()

    embeddings = []

    dataloader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        collate_fn=data_collator,
    )

    device = torch.device(
        "cuda" if torch.cuda.is_available()
        else "cpu"
    )

    model.to(device)

    with torch.no_grad():

        for batch in dataloader:

            batch = {
                k: v.to(device)
                for k, v in batch.items()
            }

            outputs = model(
                **batch,
                output_hidden_states=True,
            )

            # Last hidden layer
            hidden = outputs.hidden_states[-1]

            # Mean pooling
            attention_mask = batch["attention_mask"]

            mask = attention_mask.unsqueeze(-1)

            pooled = (
                hidden * mask
            ).sum(dim=1) / mask.sum(dim=1)

            embeddings.append(
                pooled.cpu().numpy()
            )

    return np.vstack(embeddings)


In [ ]:
train_embeddings = extract_embeddings(
    best_model,
    train_dataset,
)

dev_embeddings = extract_embeddings(
    best_model,
    validation_dataset,
)

test_embeddings = extract_embeddings(
    best_model,
    test_dataset,
)


In [ ]:
np.save(
    OUTPUT_DIR / "dev_embeddings.npy",
    dev_embeddings,
)

np.save(
    OUTPUT_DIR / "test_embeddings.npy",
    test_embeddings,
)


In [ ]:
import pandas as pd

In [ ]:
metadata = []

for i in range(len(test_dataset)):

    metadata.append({
        "index": i,
        "true_labels": test_labels[i].tolist(),
        "predicted_labels": test_predictions[i].tolist(),
    })

pd.DataFrame(metadata).to_csv(
    OUTPUT_DIR / "test_embedding_metadata.csv",
    index=False,
)


# PCA

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(
    n_components=50,
    random_state=42,
)

X_pca = pca.fit_transform(
    test_embeddings
)


In [ ]:
import umap

reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42,
)

X_umap = reducer.fit_transform(
    X_pca
)


# HDBSCAN clustering

In [ ]:
k = 10


In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=30,
    metric="euclidean",
    cluster_selection_method="eom",
)

clusters = clusterer.fit_predict(
    X_pca
)


In [ ]:
results = pd.DataFrame({
    "cluster": clusters,
})

for i, label in enumerate(model_labels):

    results[label] = test_labels[:, i]


In [ ]:
cluster_profiles = (
    results
    .groupby("cluster")[model_labels]
    .mean()
)


In [ ]:
exact_correct = np.all(
    test_labels == test_predictions,
    axis=1,
)


# Label-level correctness

In [ ]:
from sklearn.metrics import multilabel_confusion_matrix

cm = multilabel_confusion_matrix(
    test_labels,
    test_predictions,
)


In [ ]:
green = correctly classified
red   = misclassified


# Boundery documents

In [ ]:
from sklearn.metrics import pairwise_distances

distances = pairwise_distances(
    X_pca,
    metric="cosine",
)


In [ ]:
interesting_indices = np.where(
    ~exact_correct
)[0]


In [ ]:
for idx in interesting_indices[:20]:

    print("=" * 80)

    print("INDEX:", idx)

    print(
        "TRUE:",
        [
            model_labels[i]
            for i in range(len(model_labels))
            if test_labels[idx, i] == 1
        ]
    )

    print(
        "PREDICTED:",
        [
            model_labels[i]
            for i in range(len(model_labels))
            if test_predictions[idx, i] == 1
        ]
    )

    print(
        dataset["test"][idx]["text"]
    )


# Analyze register-to-register relationships

In [ ]:
centroid_A = embeddings[labels[:, A] == 1].mean(axis=0)
centroid_B = embeddings[labels[:, B] == 1].mean(axis=0)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(
    centroids
)
